# Voice Order Processing with Snowflake AI

**TastyBytes** is modernizing their phone ordering system. This notebook builds an end-to-end pipeline that goes from **raw audio recordings** to **structured order data** to **conversational analytics** powered by a Cortex Agent.

### Architecture

```
┌─────────────────┐     ┌─────────────────┐     ┌─────────────────┐
│   Order Text    │────▶│  gTTS (Python)  │────▶│   Audio Stage   │
│   (Phrases)     │     │  Generate MP3   │     │   @AUDIO_STAGE  │
└─────────────────┘     └─────────────────┘     └────────┬────────┘
                                                         │
                                                         ▼
┌─────────────────┐     ┌─────────────────┐     ┌─────────────────┐
│ CUSTOMER_ORDERS │◀────│  AI_COMPLETE    │◀────│  AI_TRANSCRIBE  │
│    (Table)      │     │  Extract JSON   │     │  Speech-to-Text │
└────────┬────────┘     └─────────────────┘     └─────────────────┘
         │
         ▼
┌─────────────────┐     ┌─────────────────┐     ┌─────────────────┐
│  Semantic View  │────▶│  Cortex Agent   │────▶│   Natural Lang  │
│   (Analytics)   │     │ (Orchestration) │     │    Queries      │
└─────────────────┘     └─────────────────┘     └─────────────────┘
```

**The journey:** Unstructured audio -> Transcribed text -> Structured JSON -> Queryable table -> Semantic View -> Cortex Agent chat

### Before You Begin
Run `setup.sql` in a SQL worksheet to create the database, schema, warehouse, stage, tables, seed data, and external access integration. Then add `GOOGLE_TTS_INTEGRATION` in your notebook's **Settings > External access** and restart the session.

In [ ]:
USE DATABASE tasty_audio_db;
USE SCHEMA orders;
SELECT COUNT(*) AS phrase_count FROM ORDER_PHRASES;

## Step 1: Generate Audio with Google Text-to-Speech

We use the **Google Translate TTS API** to convert each of the 75 order phrases into MP3 audio files. This simulates real customer phone orders that TastyBytes would receive.

The cell below does the following:
1. Queries all order phrases from the `ORDER_PHRASES` table
2. For each phrase, calls the Google Translate TTS endpoint via `requests` (enabled by the `GOOGLE_TTS_INTEGRATION` External Access Integration)
3. Writes each MP3 to an in-memory buffer using `io.BytesIO`
4. Uploads the buffer to the Snowflake internal stage `@AUDIO_STAGE` using `session.file.put_stream`
5. Files are named `order_01.mp3`, `order_02.mp3`, ..., `order_75.mp3`

In [ ]:
import requests
import io
import urllib.parse
from snowflake.snowpark.context import get_active_session

session = get_active_session()

phrases_df = session.sql("SELECT phrase_id, order_text FROM ORDER_PHRASES").collect()

for row in phrases_df:
    phrase_id = row['PHRASE_ID']
    text = row['ORDER_TEXT']
    filename = f"order_{str(phrase_id).zfill(2)}.mp3"

    encoded_text = urllib.parse.quote(text)
    url = f"https://translate.google.com/translate_tts?ie=UTF-8&client=tw-ob&tl=en&q={encoded_text}"
    response = requests.get(url)

    audio_buffer = io.BytesIO(response.content)
    session.file.put_stream(
        audio_buffer,
        f"@AUDIO_STAGE/{filename}",
        auto_compress=False,
        overwrite=True
    )
    print(f"Generated: {filename}")

print(f"\nGenerated {len(phrases_df)} audio files!")


### Verify Staged Files

`ALTER STAGE ... REFRESH` updates the directory table metadata to reflect the newly uploaded files. `LIST` displays all files in the stage with their sizes and timestamps, confirming all 75 MP3 files were uploaded.

In [ ]:
ALTER STAGE AUDIO_STAGE REFRESH;
LIST @AUDIO_STAGE;

## Step 2: Transcribe Audio with AI_TRANSCRIBE

**AI_TRANSCRIBE** is Snowflake's built-in speech-to-text function. It converts audio files into text and returns a JSON object containing the transcribed text, detected language, and duration.

Below we test on a single file first:
- `TO_FILE('@AUDIO_STAGE', 'order_01.mp3')` creates a file reference to the audio in the stage
- `AI_TRANSCRIBE` processes the audio and returns JSON like: `{"text": "I want a hamburger with pickles and onion.", "language": "en"}`

This validates the audio was generated correctly before processing all 75 files.

In [ ]:
SELECT AI_TRANSCRIBE(TO_FILE('@AUDIO_STAGE', 'order_01.mp3')) AS transcription;

## Step 3: Process All Orders with AI Pipeline

This is the **core of the solution** -- a single SQL statement that chains three AI operations using CTEs:

**CTE 1 -- `staged_files`:** Queries `DIRECTORY(@AUDIO_STAGE)` to list all MP3 files. The directory table returns file metadata including `RELATIVE_PATH`.

**CTE 2 -- `transcriptions`:** Calls `AI_TRANSCRIBE` on each audio file to convert speech to text.

**CTE 3 -- `extracted`:** Uses `AI_COMPLETE` with the `mistral-large2` model and a **JSON schema** in `response_format`. This forces the LLM to return structured output with `item_name`, `quantity`, `size`, and `special_instructions` for each order item. The schema defines `order_items` as an array to handle multi-item orders (e.g., *"2 pizzas and a salad"*).

**Final SELECT with LATERAL FLATTEN:** `PARSE_JSON(extracted_json):order_items` extracts the array, then `LATERAL FLATTEN` expands it into separate rows. This means a single audio file with 3 items produces 3 rows in `CUSTOMER_ORDERS`. `COALESCE` provides empty-string defaults for optional fields.

> **Pipeline flow:** Audio file -> AI_TRANSCRIBE (speech-to-text) -> AI_COMPLETE (text-to-JSON) -> LATERAL FLATTEN (JSON-to-rows) -> INSERT

In [ ]:
-- Insert processed orders into the CUSTOMER_ORDERS table
INSERT INTO CUSTOMER_ORDERS (
    audio_file, 
    raw_transcript, 
    item_name, 
    quantity, 
    size, 
    special_instructions
)

-- CTE 1: Get all MP3 files from the stage directory
WITH staged_files AS (
    SELECT RELATIVE_PATH AS filename
    FROM DIRECTORY(@AUDIO_STAGE)  -- Lists all files in the stage
    WHERE RELATIVE_PATH LIKE '%.mp3'  -- Filter to only MP3 files
),

-- CTE 2: Transcribe each audio file using AI_TRANSCRIBE
transcriptions AS (
    SELECT 
        filename,
        AI_TRANSCRIBE(TO_FILE('@AUDIO_STAGE', filename)) AS transcript  -- Convert speech to text
    FROM staged_files
),

-- CTE 3: Extract structured order data from transcripts using AI_COMPLETE
extracted AS (
    SELECT 
        filename,
        transcript:text::VARCHAR AS transcript_text,  -- Get the text from the transcript JSON
        AI_COMPLETE(
            model => 'mistral-large2',
            prompt => 'Extract order details from: ' || transcript:text::VARCHAR,
            -- Define the expected JSON schema for structured output
            response_format => {
                'type': 'json',
                'schema': {
                    'type': 'object',
                    'properties': {
                        'order_items': {
                            'type': 'array',
                            'items': {
                                'type': 'object',
                                'properties': {
                                    'item_name': {'type': 'string'},
                                    'quantity': {'type': 'integer'},
                                    'size': {'type': 'string'},
                                    'special_instructions': {'type': 'string'}
                                },
                                'required': ['item_name', 'quantity']
                            }
                        }
                    },
                    'required': ['order_items']
                }
            }
        ) AS extracted_json  -- AI extracts structured data from natural language
    FROM transcriptions
)

-- Final SELECT: Flatten the order_items array and map to table columns
SELECT 
    filename,
    transcript_text,
    f.value:item_name::STRING,  -- Extract item name from JSON
    f.value:quantity::INTEGER,  -- Extract quantity from JSON
    COALESCE(f.value:size::STRING, ''),  -- Extract size (default to empty string)
    COALESCE(f.value:special_instructions::STRING, '')  -- Extract instructions (default to empty)
FROM extracted,
-- LATERAL FLATTEN expands the order_items array into separate rows
-- This handles orders with multiple items (e.g., "2 pizzas and a salad")
LATERAL FLATTEN(input => PARSE_JSON(extracted_json):order_items) f;

### View Processed Orders

Query the `CUSTOMER_ORDERS` table to verify the AI pipeline correctly extracted item names, quantities, sizes, and special instructions from each audio file.

In [ ]:
SELECT * FROM CUSTOMER_ORDERS ORDER BY order_id;

## Step 4: Create Semantic View

A **Semantic View** creates a metadata layer over the `CUSTOMER_ORDERS` table that defines business meaning, enabling natural language queries through Cortex Analyst.

- **DIMENSIONS** define categorical/descriptive attributes for filtering and grouping: `item_name`, `size`, `order_status`, `order_timestamp`, `special_instructions`
- **METRICS** define aggregatable measures: `total_quantity` (SUM of quantities) and `order_count` (COUNT of distinct orders)

Once created, you can ask natural language questions like *"What are the most popular items?"* and Cortex Analyst will generate the correct SQL.

In [ ]:
CREATE OR REPLACE SEMANTIC VIEW customer_orders_semantic
  TABLES (
    orders AS tasty_audio_db.orders.CUSTOMER_ORDERS
      PRIMARY KEY (order_id)
  )
  DIMENSIONS (
    orders.item_name AS item_name,
    orders.size AS size,
    orders.order_status AS order_status,
    orders.order_timestamp AS order_timestamp,
    orders.special_instructions AS special_instructions
  )
  METRICS (
    orders.total_quantity AS SUM(quantity),
    orders.order_count AS COUNT(DISTINCT order_id)
  );

## Step 5: Analytics

Run a quick aggregation to see the most popular items by order count and total quantity. This validates that the AI pipeline correctly parsed diverse orders across burgers, pizzas, sandwiches, beverages, sides, and desserts.

In [ ]:
SELECT item_name, COUNT(*) AS order_count, SUM(quantity) AS total_qty
FROM CUSTOMER_ORDERS 
GROUP BY item_name 
ORDER BY order_count DESC;

## Step 6: Create a Cortex Agent

Now that we have structured order data and a Semantic View, the final step is to create a **Cortex Agent** that lets users ask natural language questions about orders. This completes the journey from **raw audio** to **conversational analytics**.

A Cortex Agent orchestrates across tools like **Cortex Analyst** (which uses our Semantic View) to translate natural language into SQL, execute it, and return results in plain English.

### Create the Agent in Snowsight

1. Navigate to **AI & ML > Agents** in Snowsight
2. Click **Create agent**
3. Set the **Agent object name** to `tasty_orders_agent`
4. Click **Create agent**, then click **Edit**

### Add Cortex Analyst as a Tool

1. Select the **Tools** tab
2. Find **Cortex Analyst** and click **+ Add**
3. Configure:
   - **Semantic view**: `TASTY_AUDIO_DB.ORDERS.CUSTOMER_ORDERS_SEMANTIC`
   - **Warehouse**: `TASTY_AUDIO_WH`
   - **Description**: `Analyzes customer order data including items, quantities, sizes, and special instructions`
4. Click **Add**

### Configure Orchestration

1. Select the **Orchestration** tab
2. Choose an orchestration model (e.g., `claude-3-5-sonnet`)
3. For **Planning instructions**, enter:
   ```
   Use the order_analytics tool for all questions about orders, items,
   quantities, sizes, or customer preferences.
   ```
4. Click **Save**

### Test the Agent

In the Agent Playground, try questions like:
- *"What are the top 5 most ordered items?"*
- *"How many pizza orders were there?"*
- *"Show me orders with special instructions"*
- *"What percentage of orders are for burgers?"*

The agent uses **Cortex Analyst** to convert your question into SQL against the Semantic View, runs it, and returns the answer conversationally.